# Task 2 EDA: loyalty and engagement interaction

One named dataset: eda_tier_cross_v2. Read only competition raw files. Keep 44 source features and add one unsupervised categorical cross. No statistics are fitted on Train, Validation, or Test. The two component tiers have undocumented creation times; the cross inherits that availability caveat.

In [1]:
from pathlib import Path
import sys,json
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
import numpy as np,pandas as pd
from IPython.display import display
from src.pipeline.random_seed import set_seed
from src.pipeline.save_data import save_data
SEED=set_seed()
RAW=SRC/'task2'/'data'/'raw'
NAME='eda_tier_cross_v2';TARGET='segment';ID='player_id'
train_raw=pd.read_csv(RAW/'train.csv')
test_raw=pd.read_csv(RAW/'test.csv')
assert train_raw.shape==(104000,46) and test_raw.shape==(25889,45)
assert train_raw.drop(columns=[TARGET]).columns.tolist()==test_raw.columns.tolist()
assert train_raw[ID].is_unique and test_raw[ID].is_unique
assert not set(train_raw[ID]).intersection(test_raw[ID])
assert train_raw[TARGET].notna().all() and set(train_raw[TARGET])=={0,1,2,3}
source_features=test_raw.columns.drop(ID).tolist()
assert len(source_features)==44 and not np.isinf(
    train_raw.select_dtypes(include='number').drop(columns=[TARGET]).to_numpy()).any()
assert not np.isinf(test_raw.select_dtypes(include='number').to_numpy()).any()
print('Train/Test:',train_raw.shape,test_raw.shape)

Train/Test: (104000, 46) (25889, 45)


In [2]:
def add_cross(frame):
    out=frame.copy()
    loyalty=out.loyalty_tier.astype('string').fillna('<missing>')
    engagement=out.engagement_level.astype('string').fillna('<missing>')
    out['loyalty_engagement_cross']=(loyalty+' | '+engagement).astype('object')
    return out
train=add_cross(train_raw);test=add_cross(test_raw)
train[TARGET]=train[TARGET].astype('int8')
assert train.drop(columns=[TARGET]).columns.tolist()==test.columns.tolist()
assert train[source_features].equals(train_raw[source_features])
assert test[source_features].equals(test_raw[source_features])
assert train.loyalty_engagement_cross.notna().all()
assert test.loyalty_engagement_cross.notna().all()
assert train.loyalty_engagement_cross.nunique()<=30
display(pd.crosstab(train.loyalty_engagement_cross,train.segment).head(30))
print('Cross categories train/test:',train.loyalty_engagement_cross.nunique(),
      test.loyalty_engagement_cross.nunique())

segment,0,1,2,3
loyalty_engagement_cross,,,,
<missing> | <missing>,246,165,138,97
<missing> | High,282,755,147,455
<missing> | Low,1494,212,734,119
<missing> | Mid,1161,745,582,436
<missing> | Veteran,3,155,6,93
Bronze | <missing>,1532,951,25,20
Bronze | High,1751,4349,52,65
Bronze | Low,8545,1140,98,96
Bronze | Mid,6924,4421,120,116


Cross categories train/test: 30 30


In [3]:
card_path=SRC/'task2'/'data'/'eda'/f'{NAME}.json'
if card_path.exists():
    info=json.loads(card_path.read_text(encoding='utf8'))
    assert info['name']==NAME and info['splits']['train']['rows']==len(train)
    assert info['splits']['test']['rows']==len(test)
    for split in ('train','test'):assert (SRC/info['splits'][split]['path']).exists()
    print('Loaded existing artifact:',card_path)
else:
    info=save_data(task=2,name=NAME,train=train,test=test,
        card={'source':'task2/data/raw/train.csv and test.csv',
              'preparation':'Kept all 44 raw predictors; added loyalty_engagement_cross = loyalty_tier | engagement_level, with explicit missing tokens. No fitted transform.',
              'leakage_review':'No target or player_id in cross. Tier creation times are undocumented; cross inherits their availability concern.'},
        format='parquet')
    print('Saved:',card_path)
print(info['splits']['train']['path'],info['splits']['test']['path'])

Saved: C:\Users\BTCOM\Desktop\ml-final\src\task2\data\eda\eda_tier_cross_v2.json
task2/data/eda/eda_tier_cross_v2-train.parquet task2/data/eda/eda_tier_cross_v2-test.parquet
